# D169 — Window Functions with Olist Data

D164 introduced window functions with ten simple sales records. This notebook applies non-ranking window functions to `olist_order_items`. Ranking functions remain in D165–T168.

The examples use only one source table. CTEs first prepare the correct business grain, such as one row per month or seller, before a window calculation is applied.

## 1. Connect and define the query helper

In [ ]:
import os
import mysql.connector
connection=mysql.connector.connect(
 host=os.environ.get('MYSQL_HOSTNAME','127.0.0.1'),
 port=int(os.environ.get('MYSQL_PORT','3306')),
 user=os.environ.get('MYSQL_USERNAME','root'),
 password=os.environ.get('MYSQL_PASSWORD','root'),
 database=os.environ.get('MYSQL_DATABASE','olist_import_lab'))
print('Connected:',connection.is_connected())

In [ ]:
def execute_sql(sql,max_rows=30):
 cursor=connection.cursor();cursor.execute(sql);columns=[x[0] for x in cursor.description]
 rows=cursor.fetchmany(max_rows+1);more=len(rows)>max_rows;rows=rows[:max_rows];cursor.close()
 text=[['NULL' if v is None else str(v) for v in row] for row in rows];widths=[len(c) for c in columns]
 for row in text:widths=[max(w,len(v)) for w,v in zip(widths,row)]
 print(' | '.join(c.ljust(w) for c,w in zip(columns,widths)));print('-+-'.join('-'*w for w in widths))
 for row in text:print(' | '.join(v.ljust(w) for v,w in zip(row,widths)))
 if more:print(f'... showing the first {max_rows} rows')
 return rows

## 2. Monthly running total

The `monthly` CTE creates one row per shipping month. The window then adds the current and all earlier monthly values. A running total on raw item rows would answer a different question.

In [ ]:
execute_sql("""WITH monthly AS(
 SELECT DATE_FORMAT(shipping_limit_date,'%Y-%m') shipping_month,SUM(price) item_value
 FROM olist_order_items GROUP BY DATE_FORMAT(shipping_limit_date,'%Y-%m'))
SELECT shipping_month,ROUND(item_value,2) item_value,
 ROUND(SUM(item_value) OVER(ORDER BY shipping_month
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),2) running_item_value
FROM monthly ORDER BY shipping_month""")

## 3. Three-month moving average

The frame uses the current month and two previous result rows. It smooths the monthly series. Be aware that this dataset has missing calendar months; `ROWS` means previous available rows, not necessarily consecutive calendar months.

In [ ]:
execute_sql("""WITH monthly AS(
 SELECT DATE_FORMAT(shipping_limit_date,'%Y-%m') shipping_month,SUM(price) item_value
 FROM olist_order_items GROUP BY DATE_FORMAT(shipping_limit_date,'%Y-%m'))
SELECT shipping_month,ROUND(item_value,2) item_value,
 ROUND(AVG(item_value) OVER(ORDER BY shipping_month
  ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),2) three_month_average
FROM monthly ORDER BY shipping_month""")

## 4. Month-over-month change with `LAG`

`LAG` brings the previous monthly value onto the current row. The outer query calculates absolute and percentage change. `NULLIF` prevents division by zero.

In [ ]:
execute_sql("""WITH monthly AS(
 SELECT DATE_FORMAT(shipping_limit_date,'%Y-%m') shipping_month,SUM(price) item_value
 FROM olist_order_items GROUP BY DATE_FORMAT(shipping_limit_date,'%Y-%m')),
compared AS(SELECT *,LAG(item_value) OVER(ORDER BY shipping_month) previous_value FROM monthly)
SELECT shipping_month,ROUND(item_value,2) item_value,ROUND(previous_value,2) previous_value,
 ROUND(item_value-previous_value,2) value_change,
 ROUND(100*(item_value-previous_value)/NULLIF(previous_value,0),2) change_percent
FROM compared ORDER BY shipping_month""")

## 5. Look ahead with `LEAD`

`LEAD` displays the next available month and its value. This can support forward comparisons or checks for gaps in a sequence.

In [ ]:
execute_sql("""WITH monthly AS(
 SELECT DATE_FORMAT(shipping_limit_date,'%Y-%m') shipping_month,SUM(price) item_value
 FROM olist_order_items GROUP BY DATE_FORMAT(shipping_limit_date,'%Y-%m'))
SELECT shipping_month,ROUND(item_value,2) item_value,
 LEAD(shipping_month) OVER(ORDER BY shipping_month) next_available_month,
 ROUND(LEAD(item_value) OVER(ORDER BY shipping_month),2) next_month_value
FROM monthly ORDER BY shipping_month""")

## 6. Seller contribution to all item value

The CTE creates one row per seller. `SUM(item_value) OVER()` calculates the grand total without collapsing seller rows.

In [ ]:
execute_sql("""WITH seller_totals AS(
 SELECT seller_id,COUNT(*) items_sold,SUM(price) item_value
 FROM olist_order_items GROUP BY seller_id)
SELECT seller_id,items_sold,ROUND(item_value,2) item_value,
 ROUND(100*item_value/SUM(item_value) OVER(),3) percent_of_all_item_value
FROM seller_totals ORDER BY item_value DESC LIMIT 15""")

## 7. Compare an item with its seller average

`PARTITION BY seller_id` calculates a separate average for each seller while preserving item rows. A CTE is needed before filtering on the window result.

In [ ]:
execute_sql("""WITH compared AS(
 SELECT order_id,order_item_id,seller_id,price,
  AVG(price) OVER(PARTITION BY seller_id) seller_average
 FROM olist_order_items)
SELECT order_id,order_item_id,seller_id,price,ROUND(seller_average,2) seller_average,
 ROUND(price-seller_average,2) difference_from_average
FROM compared WHERE price>=seller_average*5
ORDER BY difference_from_average DESC LIMIT 15""")

## 8. First and last item price for each seller

Items are ordered by shipping deadline. `LAST_VALUE` uses the full partition frame; otherwise it can return the current row's value instead of the final seller value.

In [ ]:
execute_sql("""SELECT seller_id,shipping_limit_date,price,
 FIRST_VALUE(price) OVER(PARTITION BY seller_id
  ORDER BY shipping_limit_date,order_id,order_item_id) first_item_price,
 LAST_VALUE(price) OVER(PARTITION BY seller_id
  ORDER BY shipping_limit_date,order_id,order_item_id
  ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) last_item_price
FROM olist_order_items
WHERE seller_id='4869f7a5dfa277a7dca6462dcf3b52b2'
ORDER BY shipping_limit_date,order_id,order_item_id LIMIT 15""")

## 9. Cumulative freight within an order

The partition restarts for each order. The frame follows item sequence so each row shows freight accumulated through that item. Only orders with at least five items are selected first.

In [ ]:
execute_sql("""WITH large_orders AS(
 SELECT order_id FROM olist_order_items GROUP BY order_id HAVING COUNT(*)>=5)
SELECT i.order_id,i.order_item_id,i.freight_value,
 SUM(i.freight_value) OVER(PARTITION BY i.order_id ORDER BY i.order_item_id
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) cumulative_freight
FROM olist_order_items i JOIN large_orders l ON l.order_id=i.order_id
ORDER BY i.order_id,i.order_item_id LIMIT 25""")

## 10. Running share of total value

Sellers are ordered from largest to smallest. One window calculates the running value and another calculates the grand total. Their ratio shows how quickly seller value accumulates.

In [ ]:
execute_sql("""WITH seller_totals AS(
 SELECT seller_id,SUM(price) item_value FROM olist_order_items GROUP BY seller_id),
accumulated AS(SELECT *,
 SUM(item_value) OVER(ORDER BY item_value DESC,seller_id
  ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) running_value,
 SUM(item_value) OVER() all_value FROM seller_totals)
SELECT seller_id,ROUND(item_value,2) item_value,
 ROUND(running_value,2) running_value,
 ROUND(100*running_value/all_value,2) cumulative_value_percent
FROM accumulated ORDER BY item_value DESC,seller_id LIMIT 20""")

## 11. Practical reminders

- Prepare the correct grain before applying a window.
- `PARTITION BY` restarts a calculation; it does not remove detail rows.
- Window `ORDER BY` controls calculation sequence; final `ORDER BY` controls display.
- Use explicit `ROWS` frames for running and moving calculations.
- Add stable tie-break columns when sequence must be deterministic.
- Remember that `LAST_VALUE` often needs `UNBOUNDED FOLLOWING`.
- Use a CTE before filtering a calculated window value.
- A previous available row is not always the previous calendar period. Build a complete calendar when missing periods matter.

In [ ]:
connection.close()
print('MySQL connection closed.')